In [1]:
import pandas as pd

In [6]:
raw_curated_sheet=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260506_CuratedSplicingTruthset.xlsx', sheet_name='curated_variants')
first_pass_vep=pd.read_csv('./Data/sge_data_for_qc/spliceai_benchmarking/20260507_CuratedSplicing_VEPAnnotations_firstpass.txt', sep='\t')

In [7]:
#initial merge

NM_TO_GENE = {
    "NM_002878": "RAD51D", "NM_058216": "RAD51C",
    "NM_007294": "BRCA1",  "NM_000059": "BRCA2",
    "NM_000051": "ATM",    "NM_007194": "CHEK2",
    "NM_024675": "PALB2",  "NM_004360": "CDH1",
    "NM_000249": "MLH1",   "NM_000251": "MSH2",
    "NM_000179": "MSH6",   "NM_000535": "PMS2",
    "NM_005732": "RAD50",  "NM_032043": "BRIP1",
    "NM_000038": "APC",    "NM_000546": "TP53",
    "NM_000143": "FH",     "NM_006767": "LZTR1",
    "NM_000297": "PKD2",   "NM_000314": "PTEN",
    "NM_000465": "BARD1",  "NM_182625": "GEN1",
}

GENE_TO_NM = {
    "RAD51D": "NM_002878.4", "RAD51C": "NM_058216.3",
    "BRCA1":  "NM_007294.4", "BRCA2":  "NM_000059.4",
    "ATM":    "NM_000051.4", "CHEK2":  "NM_007194.4",
    "PALB2":  "NM_024675.4", "CDH1":   "NM_004360.5",
    "MLH1":   "NM_000249.4", "MSH2":   "NM_000251.3",
    "MSH6":   "NM_000179.3", "PMS2":   "NM_000535.7",
    "RAD50":  "NM_005732.4", "BRIP1":  "NM_032043.3",
    "APC":    "NM_000038.6", "TP53":   "NM_000546.6",
    "FH":     "NM_000143.4", "LZTR1":  "NM_006767.4",
    "PKD2":   "NM_000297.4", "PTEN":   "NM_000314.8",
    "BARD1":  "NM_000465.4", "GEN1":   "NM_182625.3",
}

DROP_VARIANTS = {
    ("BRCA2", "c.7397C>T"),
    ("BRCA1", "c.190G>T"),
    ("TP53",  "c.786-60G>A"),
    ("ATM",   "c.5007-3T>A"),
    ("ATM",   "c.8787-13G>T"),
}

SPLICEAI_COLS = [
    "SpliceAI_pred_DP_AG", "SpliceAI_pred_DP_AL",
    "SpliceAI_pred_DP_DG", "SpliceAI_pred_DP_DL",
    "SpliceAI_pred_DS_AG", "SpliceAI_pred_DS_AL",
    "SpliceAI_pred_DS_DG", "SpliceAI_pred_DS_DL",
    "SpliceAI_pred_SYMBOL",
]

# --- assumes raw_curated_sheet (main sheet) and vep (VEP output) are already loaded ---
# vep should be loaded skipping ## comment lines e.g.:
# with open("vep_output.txt") as f:
#     lines = [l for l in f if not l.startswith("##")]
# vep = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
# vep.columns = [c.lstrip("#") for c in vep.columns]

# clean main raw_curated_sheet
raw_curated_sheet["Gene"]   = raw_curated_sheet["Gene"].str.strip()
raw_curated_sheet["hgvs_c"] = raw_curated_sheet["hgvs_c"].str.strip()
raw_curated_sheet = raw_curated_sheet[raw_curated_sheet["Gene"].notna() & raw_curated_sheet["hgvs_c"].notna()].copy()

# drop problem variants
raw_curated_sheet = raw_curated_sheet[~raw_curated_sheet.apply(lambda r: (r["Gene"], r["hgvs_c"]) in DROP_VARIANTS, axis=1)].copy()

# build merge key
raw_curated_sheet["hgvs_full"] = raw_curated_sheet.apply(
    lambda r: f"{GENE_TO_NM[r['Gene']]}:{r['hgvs_c'].replace(' ', '')}"
    if r["Gene"] in GENE_TO_NM else None, axis=1
)

# filter VEP to correct gene only (removes off-target neighbors)
first_pass_vep["expected_gene"] = first_pass_vep["Uploaded_variation"].apply(
    lambda x: NM_TO_GENE.get(str(x).split(":")[0].rsplit(".", 1)[0])
)
vep_clean =first_pass_vep[first_pass_vep["SYMBOL"] == first_pass_vep["expected_gene"]].copy()

# deduplicate and select columns
keep = ["Uploaded_variation", "Location", "HGVSp"] + SPLICEAI_COLS
vep_slim = (vep_clean[keep]
            .drop_duplicates(subset="Uploaded_variation")
            .rename(columns={"Uploaded_variation": "hgvs_full",
                             "Location": "pos_vep",
                             "HGVSp": "hgvsp_vep"}))

# merge
merged = raw_curated_sheet.merge(vep_slim, on="hgvs_full", how="left")

# fill pos
merged["pos"] = merged["pos_vep"]

# fill aa: prefer existing, fall back to VEP HGVSp (strip transcript prefix)
existing_aa = merged["amino_acid_substitution"].fillna("").str.strip()
merged["amino_acid_substitution"] = existing_aa.where(
    existing_aa != "",
    merged["hgvsp_vep"].apply(
        lambda x: x.split(":")[-1] if pd.notna(x) and ":" in str(x) else (None if pd.isna(x) else x)
    )
)


merged.head()

,Gene,pos,cds_pos,hgvs_c,hgvs_full,amino_acid_substitution,splice_consequence,source,pos_vep,hgvsp_vep,SpliceAI_pred_DP_AG,SpliceAI_pred_DP_AL,SpliceAI_pred_DP_DG,SpliceAI_pred_DP_DL,SpliceAI_pred_DS_AG,SpliceAI_pred_DS_AL,SpliceAI_pred_DS_DG,SpliceAI_pred_DS_DL,SpliceAI_pred_SYMBOL
0,RAD51D,17:35118621-35118621,NaN,c.145-2A>G,NM_002878.4:c.145-2A>G,-,abnormal,bueno_martinez_2021,17:35118621-35118621,-,-,-,-,-,-,-,-,-,-
1,RAD51D,17:35118495-35118495,NaN,c.263+6T>C,NM_002878.4:c.263+6T>C,-,normal,bueno_martinez_2021,17:35118495-35118495,-,-,-,-,-,-,-,-,-,-
2,RAD51D,17:35107368-35107368,NaN,c.343C>T,NM_002878.4:c.343C>T,-,abnormal,bueno_martinez_2021,17:35107368-35107368,-,-,-,-,-,-,-,-,-,-
3,RAD51D,17:35107364-35107364,NaN,c.345+2T>C,NM_002878.4:c.345+2T>C,-,abnormal,bueno_martinez_2021,17:35107364-35107364,-,-,-,-,-,-,-,-,-,-
4,RAD51D,17:35106987-35106987,NaN,c.480+1G>A,NM_002878.4:c.480+1G>A,-,abnormal,bueno_martinez_2021,17:35106987-35106987,-,-,-,-,-,-,-,-,-,-
